# Evaluating the four methods

Nothing here is deployed. Each query notebook already ran the same questions through its own `ask()` and saved what happened to a file. This notebook only **reads those files and grades them**, with one judge, so every method is graded the same way.

```
data/eval/eval_questions.jsonl
   ├─► OKF_Bundle/OKF_query.ipynb      ─► artifacts/results/okf.jsonl           ┐
   ├─► 03_graph_query.ipynb            ─► artifacts/results/graph.jsonl         │
   ├─► 05_graph_query_hybrid.ipynb     ─► artifacts/results/graph_hybrid.jsonl  ├─► this notebook ─► scores.jsonl ─► tables
   └─► 06_okf_hybrid_query.ipynb       ─► artifacts/results/okf_hybrid.jsonl    ┘
```

Each results line has the question, the **answer**, the **`retrieval_context`** (the exact blocks the model saw), timings and token counts. The questions file has the **reference answer** (`expected_output`) and where it comes from in the papers.

**Run order:** `04_graph_communities` (once) → the save-results cell at the end of each of the four query notebooks → this notebook.

| | keyword only (BM25) | hybrid (vectors + BM25 + RRF) |
|---|---|---|
| **OKF** (curated wiki + links) | `okf` | `okf_hybrid` |
| **Knowledge graph** (facts + communities) | `graph` | `graph_hybrid` |

## 1. Setup

The only real decision here is **the judge**: the model that reads each answer and context and says yes/no to DeepEval's questions.

| `JUDGE_PROVIDER` | `JUDGE_MODEL` | notes |
|---|---|---|
| `"ollama"` | `"qwen3:4b-instruct-2507-q4_K_M"` | default: free, already installed. But it's the same model that wrote the answers, so it's grading its own work, and a 4B model misses subtle mistakes. Fine for a first pass. |
| `"ollama"` | `"qwen3:8b"` | `ollama pull qwen3:8b` first. Bigger and different from the answering model. With 6 GB VRAM part of it runs on the CPU, so it's slower. |
| `"openai"` | `"gpt-4.1-mini"` (or similar) | needs `OPENAI_API_KEY` in `.env`. Strongest and fastest, costs a little per call. |

Whatever you pick, **use the same judge for all four methods**. Scores from different judges aren't comparable, which is why `scores.jsonl` keeps the judge's name on every line.

`QUESTION_LIMIT = 3` is a good first run, to see how long one question takes on your machine before starting all of them.

In [ ]:
import asyncio
import hashlib
import json
import os
import re
import time
from pathlib import Path

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

import pandas as pd
import requests
from dotenv import load_dotenv
from deepeval.metrics import (AnswerRelevancyMetric, ContextualPrecisionMetric,
                              ContextualRecallMetric, FaithfulnessMetric)
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

EVAL_PATH = PROJECT_ROOT / "data" / "eval" / "eval_questions.jsonl"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "results"
SCORES_PATH = RESULTS_DIR / "scores.jsonl"

METHODS = {                      # results file name -> label in the tables
    "okf": "OKF",
    "okf_hybrid": "OKF hybrid",
    "graph": "Graph",
    "graph_hybrid": "Graph hybrid",
}

# the judge: one for every method
JUDGE_PROVIDER = "ollama"                       # "ollama" or "openai"
JUDGE_MODEL = "qwen3:4b-instruct-2507-q4_K_M"   # or "qwen3:8b", or e.g. "gpt-4.1-mini" with "openai"
OLLAMA_URL = "http://localhost:11434"
JUDGE_NUM_CTX = 12288        # contexts are up to ~18,000 characters (~4,500 tokens) plus DeepEval's instructions
JUDGE_MAX_TOKENS = 4096      # cap per judge reply, so a model stuck repeating itself can't run forever

INCLUDE_REASON = False       # True = the judge also writes a sentence explaining each score (one more call per metric)
THRESHOLD = 0.5              # only used for DeepEval's pass/fail flag; the tables use the raw scores
QUESTION_LIMIT = None        # e.g. 3 for a quick trial run

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("project:", PROJECT_ROOT)
print("judge:", JUDGE_PROVIDER, JUDGE_MODEL)

## 2. Load the questions and the four results files

A results line counts only if its question text still matches the questions file. If you edit a question (or add new ones), re-run the save-results cell in each query notebook, otherwise that question shows up as **stale** or **missing** below.

In [ ]:
questions = [json.loads(line) for line in EVAL_PATH.open(encoding="utf-8") if line.strip()]
by_id = {q["id"]: q for q in questions}
assert len(by_id) == len(questions), "question ids must be unique"

results = {}
for method in METHODS:
    path = RESULTS_DIR / f"{method}.jsonl"
    if path.exists():
        results[method] = {r["id"]: r for r in (json.loads(line) for line in path.open(encoding="utf-8") if line.strip())}
    else:
        results[method] = {}
        print(f"missing {path.name}: run the save-results cell in that notebook")


def result_for(method, q):
    r = results[method].get(q["id"])
    return r if r is not None and r["question"] == q["question"] else None


def status(method, q):
    if q["id"] not in results[method]:
        return "missing"
    return "ok" if result_for(method, q) else "stale"


coverage = pd.DataFrame({label: [status(m, q) for q in questions] for m, label in METHODS.items()},
                        index=[q["id"] for q in questions])
print(f"{len(questions)} questions: {sum(q['answerable'] for q in questions)} answerable, "
      f"{sum(not q['answerable'] for q in questions)} unanswerable")
print(pd.Series([q["type"] for q in questions]).value_counts().to_string(), "\n")
print(coverage.apply(pd.Series.value_counts).fillna(0).astype(int).to_string())
problems = coverage[(coverage != "ok").any(axis=1)]
if len(problems):
    print("\nnot ok:\n" + problems.to_string())

## 3. The four metrics

Two of them grade **retrieval** (did the method find the right material?) and two grade **generation** (did the model use it well?). That split is what tells you *why* a method loses: it never found the facts, or it found them and wrote a poor answer.

| metric | grades | uses | what the judge is asked |
|---|---|---|---|
| **Contextual precision** | retrieval | question, reference answer, context blocks | for each block, in order: is it useful for the reference answer? The score is high when the useful blocks come **first**. |
| **Contextual recall** | retrieval | reference answer, context blocks | for each sentence of the reference answer: can it be found in the context? |
| **Faithfulness** | generation | answer, context blocks | pull the claims out of the answer; is each one supported by the context? |
| **Answer relevancy** | generation | question, answer | split the answer into statements; does each one address the question? |

All four are 0–1, higher is better. Each one is a few judge calls (extract, then give verdicts), so one question × one method ≈ 6–10 calls.

Two things these four **don't** do:
- None compares the answer with the reference directly. Recall checks the *context* against the reference; faithfulness checks the *answer* against the context. An answer can be faithful, relevant and still leave out half of what was found.
- Precision counts **blocks**, and the methods' blocks differ in size: OKF blocks are whole sections (up to 4,500 characters), graph blocks are chunks plus one block of facts. More, smaller blocks give more chances to rank useless ones high.

**Unanswerable questions** (the facts aren't in the papers) skip DeepEval. They're graded by one rule: did the method refuse?

## 4. The judge

DeepEval needs a model behind every metric. For OpenAI it has one built in. For Ollama we write a small wrapper (`DeepEvalBaseLLM` needs `load_model`, `generate`, `a_generate`, `get_model_name`) so we control three things:

- **`temperature: 0`**: the same answer gets the same verdict every time you run this.
- **`format` = the JSON schema DeepEval expects.** DeepEval asks for answers as JSON (lists of claims, yes/no verdicts). Small models often break JSON; Ollama's structured output forces valid JSON in the right shape. If it still fails to parse, the raw text goes back to DeepEval, which tries to repair it.
- **`num_ctx`**: Ollama's default context window would cut off the long contexts. `num_predict` caps each reply.

In [ ]:
class OllamaJudge(DeepEvalBaseLLM):
    # DeepEval judge backed by a local Ollama model

    def __init__(self, model_name, url=OLLAMA_URL, num_ctx=JUDGE_NUM_CTX, max_tokens=JUDGE_MAX_TOKENS, timeout=900):
        self.model_name = model_name
        self.url = url.rstrip("/")
        self.num_ctx = num_ctx
        self.max_tokens = max_tokens
        self.timeout = timeout
        self.calls, self.seconds, self.json_fallbacks = 0, 0.0, 0
        super().__init__(model_name)

    def load_model(self):
        return self.model_name

    def get_model_name(self):
        return f"ollama/{self.model_name}"

    def _chat(self, prompt, schema=None):
        body = {"model": self.model_name, "stream": False, "keep_alive": "30m",
                "messages": [{"role": "user", "content": prompt}],
                "options": {"temperature": 0, "num_ctx": self.num_ctx, "num_predict": self.max_tokens}}
        if schema is not None:
            body["format"] = schema.model_json_schema()
        start = time.perf_counter()
        resp = requests.post(f"{self.url}/api/chat", json=body, timeout=self.timeout)
        if resp.status_code >= 400 and schema is not None:     # older Ollama without JSON-schema output: plain JSON mode
            body["format"] = "json"
            resp = requests.post(f"{self.url}/api/chat", json=body, timeout=self.timeout)
        resp.raise_for_status()
        self.calls += 1
        self.seconds += time.perf_counter() - start
        text = resp.json()["message"]["content"]
        return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()    # in case a thinking model is used

    def generate(self, prompt, schema=None, **kwargs):
        text = self._chat(prompt if isinstance(prompt, str) else str(prompt), schema)
        if schema is None:
            return text
        try:
            return schema.model_validate_json(text)
        except Exception:
            self.json_fallbacks += 1
            return text                                         # DeepEval tries to repair it

    async def a_generate(self, prompt, schema=None, **kwargs):
        return await asyncio.to_thread(self.generate, prompt, schema)


if JUDGE_PROVIDER == "ollama":
    judge = OllamaJudge(JUDGE_MODEL)
    JUDGE_NAME = judge.get_model_name()
elif JUDGE_PROVIDER == "openai":
    assert os.getenv("OPENAI_API_KEY"), "put OPENAI_API_KEY in .env"
    judge = JUDGE_MODEL                     # DeepEval's built-in OpenAI judge
    JUDGE_NAME = f"openai/{JUDGE_MODEL}"
else:
    raise ValueError(JUDGE_PROVIDER)

# smoke test: one structured call
if JUDGE_PROVIDER == "ollama":
    from pydantic import BaseModel

    class Ping(BaseModel):
        verdict: str
        reason: str

    t0 = time.perf_counter()
    print(judge.generate('Is Paris the capital of France? Reply as JSON with "verdict" ("yes" or "no") and "reason".',
                         schema=Ping))
    print(f"{time.perf_counter() - t0:.1f} s (the first call also loads the model)")

## 5. Refusals

All four methods are told to say "The bundle / graph / knowledge **does not contain this.**" when the context doesn't have the answer. That gives two checks with no judge at all:

- **unanswerable questions**: refusing is the right answer.
- **answerable questions**: a refusal means the method missed something it should have found.

A refusal counts as **full** when the answer is only that sentence, and **partial** when it comes with other text (common on two-part questions, where one half is answered).

A full refusal has no claims, so DeepEval would give it a perfect faithfulness of 1.0. To keep that from flattering a method, **faithfulness is skipped for full refusals**. Answer relevancy still scores them (low), and the retrieval metrics don't look at the answer at all.

In [ ]:
REFUSAL_RE = re.compile(r"\b(bundle|graph|knowledge|context)\s+(does not|doesn't|do not)\s+contain\s+this", re.I)


def refusal(answer):
    if not REFUSAL_RE.search(answer):
        return None
    rest = re.sub(r"\[[^\]]*\]|[\s.,;:]", "", REFUSAL_RE.sub("", answer))   # drop the sentence, citations, punctuation
    return "full" if len(rest) < 40 else "partial"


rows = []
for q in questions:
    for method, label in METHODS.items():
        r = result_for(method, q)
        if r:
            rows.append({"method": label, "answerable": q["answerable"], "refusal": refusal(r["answer"])})
ref = pd.DataFrame(rows)
table = pd.DataFrame({
    "answerable: full refusals": ref[ref.answerable].groupby("method").refusal.apply(lambda s: f"{(s == 'full').sum()}/{len(s)}"),
    "answerable: partial": ref[ref.answerable].groupby("method").refusal.apply(lambda s: f"{(s == 'partial').sum()}/{len(s)}"),
    "unanswerable: refused (good)": ref[~ref.answerable].groupby("method").refusal.apply(lambda s: f"{s.notna().sum()}/{len(s)}"),
}).reindex([l for l in METHODS.values() if l in set(ref.method)])
print(table.to_string())

## 6. Score everything

For every answerable question and every method, this builds a DeepEval `LLMTestCase` from the files and runs the four metrics:

```python
LLMTestCase(input=question, actual_output=answer, expected_output=reference_answer, retrieval_context=blocks)
```

How the loop is set up:
- **Question by question, all four methods each time.** If you stop it halfway, the methods have been graded on the same questions, so the tables below still compare like with like. They only use questions that every method has scores for.
- **Checkpointed.** Every score is appended to `artifacts/results/scores.jsonl` as soon as it exists. Re-running skips what's done, so you can stop and continue later.
- **Fingerprinted.** Each score remembers a hash of the question, reference, answer and context. Re-run a method's notebook and its changed answers get re-graded; unchanged ones don't.
- **Errors are kept and retried.** If the judge returns something DeepEval can't parse, that score is saved as an error and tried again on the next run.

Time: a 4B judge on your GPU takes roughly 5–20 s per call, and there are ~27 answerable questions × 4 methods × 6–10 calls. Plan on a couple of hours for the full set; start with `QUESTION_LIMIT = 3`.

In [ ]:
METRICS = {
    "contextual_precision": ContextualPrecisionMetric,
    "contextual_recall": ContextualRecallMetric,
    "faithfulness": FaithfulnessMetric,
    "answer_relevancy": AnswerRelevancyMetric,
}
SHORT = {"contextual_precision": "prec", "contextual_recall": "rec", "faithfulness": "faith", "answer_relevancy": "rel"}


def fingerprint(q, r):
    blob = json.dumps([q["question"], q["expected_output"], r["answer"], r["retrieval_context"]], ensure_ascii=False)
    return hashlib.sha1(blob.encode("utf-8")).hexdigest()[:12]


def load_scores():
    scores = {}
    if SCORES_PATH.exists():
        for line in SCORES_PATH.open(encoding="utf-8"):
            if line.strip():
                s = json.loads(line)
                scores[(s["judge"], s["method"], s["id"], s["metric"])] = s     # later lines win
    return scores


def measure(name, q, r):
    test_case = LLMTestCase(input=q["question"], actual_output=r["answer"], expected_output=q["expected_output"],
                            retrieval_context=r["retrieval_context"])
    metric = METRICS[name](threshold=THRESHOLD, model=judge, include_reason=INCLUDE_REASON,
                           async_mode=False, verbose_mode=False)
    metric.measure(test_case, _show_indicator=False)
    return float(metric.score), metric.reason


def fmt(x):
    return " -- " if x is None else f"{x:.2f}"

In [ ]:
scores = load_scores()
todo = [q for q in questions if q["answerable"]][:QUESTION_LIMIT]
calls_before = judge.calls if isinstance(judge, OllamaJudge) else 0
run_start, n_new = time.perf_counter(), 0

with SCORES_PATH.open("a", encoding="utf-8") as out:
    for n, q in enumerate(todo, start=1):
        for method, label in METHODS.items():
            r = result_for(method, q)
            if r is None:
                continue
            fp, full_refusal = fingerprint(q, r), refusal(r["answer"]) == "full"
            line, t0 = [], time.perf_counter()
            for name in METRICS:
                key = (JUDGE_NAME, method, q["id"], name)
                old = scores.get(key)
                if old and old["fingerprint"] == fp and old["error"] is None:
                    line.append(f"{SHORT[name]} {fmt(old['score'])}")
                    continue
                rec = {"judge": JUDGE_NAME, "method": method, "id": q["id"], "metric": name, "fingerprint": fp,
                       "score": None, "reason": None, "error": None, "skipped": None, "seconds": 0.0}
                if name == "faithfulness" and full_refusal:
                    rec["skipped"] = "full refusal"
                else:
                    t1 = time.perf_counter()
                    try:
                        rec["score"], rec["reason"] = measure(name, q, r)
                    except Exception as e:
                        rec["error"] = f"{type(e).__name__}: {e}"[:300]
                    rec["seconds"] = round(time.perf_counter() - t1, 1)
                    n_new += 1
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                out.flush()
                scores[key] = rec
                line.append(f"{SHORT[name]} {'ERR ' if rec['error'] else fmt(rec['score'])}")
            print(f"[{n:2}/{len(todo)}] {q['id']:4} {label:12} " + " | ".join(line) + f"   ({time.perf_counter() - t0:.0f} s)")

elapsed = time.perf_counter() - run_start
print(f"\n{n_new} new scores in {elapsed / 60:.1f} min", end="")
if isinstance(judge, OllamaJudge):
    calls = judge.calls - calls_before
    print(f" | {calls} judge calls, {judge.seconds / max(judge.calls, 1):.1f} s each, "
          f"{judge.json_fallbacks} JSON repairs", end="")
errors = sum(1 for s in scores.values() if s["judge"] == JUDGE_NAME and s["error"])
print(f" | {errors} errors (re-run this cell to retry them)")

## 7. Results

One row per method. The four metrics are averaged over the answerable questions that **every** method has scores for. The other columns:

- **answered %**: answerable questions that didn't get a full refusal.
- **unanswerable refused**: how many unanswerable questions were (correctly) refused.
- **retrieval / generation ms**: medians from the results files. These come from your machine, so they're comparable only because all four notebooks ran on it with the same model.
- **prompt / answer tokens**: means. Prompt tokens is the cost of the context.

Small differences are noise: with ~27 questions, the ± table under the summary shows how far each average could move by chance. If two methods' ranges overlap a lot, don't call a winner on that metric.

In [ ]:
scores = load_scores()
rows = []
for q in questions:
    for method, label in METHODS.items():
        r = result_for(method, q)
        if r is None:
            continue
        fp = fingerprint(q, r)
        row = {"id": q["id"], "type": q["type"], "papers": q["papers"], "answerable": q["answerable"], "method": label,
               "refusal": refusal(r["answer"]), "retrieval_ms": r["retrieval_ms"], "generation_ms": r["generation_ms"],
               "prompt_tokens": r["prompt_tokens"], "answer_tokens": r["answer_tokens"],
               "context_blocks": len(r["retrieval_context"]), "context_chars": sum(map(len, r["retrieval_context"]))}
        for name in METRICS:
            s = scores.get((JUDGE_NAME, method, q["id"], name))
            ok = s is not None and s["fingerprint"] == fp and s["error"] is None
            row[name] = s["score"] if ok else None
            row[name + "_done"] = ok
        rows.append(row)
df = pd.DataFrame(rows)
METRIC_COLS = list(METRICS)
LABELS = [l for l in METHODS.values() if l in set(df.method)]

done_all = df[df.answerable].groupby("id")[[m + "_done" for m in METRIC_COLS]].all().all(axis=1)
done_all = done_all[done_all].index
per_method = df.groupby("id").method.nunique()
common = [i for i in done_all if per_method[i] == len(METHODS)]
graded = df[df.answerable & df.id.isin(common)]
unans = df[~df.answerable]
print(f"graded by {JUDGE_NAME}: {len(common)} answerable questions with all four methods scored")

summary = pd.DataFrame({
    **{m: graded.groupby("method")[m].mean() for m in METRIC_COLS},
    "answered %": df[df.answerable].groupby("method").refusal.apply(lambda s: 100 * (s != "full").mean()),
    "unanswerable refused": unans.groupby("method").refusal.apply(lambda s: f"{s.notna().sum()}/{len(s)}"),
    "retrieval ms": df.groupby("method").retrieval_ms.median(),
    "generation ms": df.groupby("method").generation_ms.median(),
    "prompt tokens": df.groupby("method").prompt_tokens.mean(),
    "answer tokens": df.groupby("method").answer_tokens.mean(),
}).reindex(LABELS)

margin = pd.DataFrame({m: 1.96 * graded.groupby("method")[m].std() / graded.groupby("method")[m].count() ** 0.5
                       for m in METRIC_COLS}).reindex(LABELS)

try:
    display(summary.style.format(precision=3).format(precision=0, subset=["answered %", "retrieval ms", "generation ms",
                                                                          "prompt tokens", "answer tokens"])
            .highlight_max(subset=METRIC_COLS, color="#d4edda"))
except Exception:
    display(summary.round(3))
print("± (95% range of each average):")
display(margin.round(3))

In [ ]:
# the same four averages as a chart
try:
    import matplotlib.pyplot as plt

    ax = summary[METRIC_COLS].T.plot.bar(figsize=(9, 4), ylim=(0, 1), rot=0,
                                          yerr=margin[METRIC_COLS].T, capsize=3)
    ax.set_title(f"four methods, judged by {JUDGE_NAME}")
    ax.set_ylabel("mean score (0-1)")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; the table above has the same numbers")

### By question type

The averages above hide where each method wins. Retrieval styles tend to split by type: keyword search is good at exact numbers and names, vectors at paraphrased concepts, graphs and links at questions that span papers.

In [ ]:
for m in METRIC_COLS:
    print(f"\n{m}")
    display(graded.pivot_table(index="type", columns="method", values=m, aggfunc="mean").reindex(columns=LABELS).round(2))

### By paper

Contextual recall per paper: does a method find facts in every paper, or does it lean on some? (A question about several papers counts for each of them.)

In [ ]:
by_paper = graded.explode("papers")
display(by_paper.pivot_table(index="papers", columns="method", values="contextual_recall", aggfunc="mean")
        .reindex(columns=LABELS).round(2))

### Question by question

One metric at a time (change `SHOW_METRIC`), sorted with the hardest questions on top. Use `show_question("q07")` below to see the answers, scores and context side by side.

In [ ]:
SHOW_METRIC = "contextual_recall"

wide = graded.pivot_table(index=["id", "type"], columns="method", values=SHOW_METRIC, aggfunc="mean").reindex(columns=LABELS)
wide["mean"] = wide.mean(axis=1)
display(wide.sort_values("mean").round(2))

In [ ]:
def show_question(qid, answer_chars=700):
    q = by_id[qid]
    print(f"{qid} [{q['type']}] {q['question']}\n")
    print("REFERENCE:", q["expected_output"], "\n")
    for method, label in METHODS.items():
        r = result_for(method, q)
        if r is None:
            print(f"=== {label}: no result\n")
            continue
        fp = fingerprint(q, r)
        line = []
        for name in METRICS:
            s = scores.get((JUDGE_NAME, method, qid, name))
            ok = s is not None and s["fingerprint"] == fp
            line.append(f"{SHORT[name]} {fmt(s['score']) if ok else ' -- '}")
        print(f"=== {label}: " + " | ".join(line) + f" | refusal: {refusal(r['answer'])}")
        print(r["answer"][:answer_chars] + (" ..." if len(r["answer"]) > answer_chars else ""))
        print("context:", " / ".join(block.strip().splitlines()[0][:60] for block in r["retrieval_context"]))
        if INCLUDE_REASON:
            for name in METRICS:
                s = scores.get((JUDGE_NAME, method, qid, name))
                if s and s.get("reason"):
                    print(f"  {SHORT[name]}: {s['reason']}")
        print()


show_question(questions[0]["id"])

In [ ]:
# unanswerable questions: what each method said
for q in questions:
    if q["answerable"]:
        continue
    print(f"{q['id']} {q['question']}")
    for method, label in METHODS.items():
        r = result_for(method, q)
        if r:
            verdict = "refused ✓" if refusal(r["answer"]) else "ANSWERED ✗"
            print(f"   {label:12} {verdict:11} {r['answer'][:110]!r}")
    print()

## 8. Save the tables

In [ ]:
summary.to_csv(RESULTS_DIR / "summary.csv")
df.drop(columns=[m + "_done" for m in METRIC_COLS]).to_csv(RESULTS_DIR / "per_question_scores.csv", index=False)
print("saved summary.csv and per_question_scores.csv to", RESULTS_DIR)

## How far to trust these numbers

- **The judge is a model too.** A 4B judge misses subtle errors and can be inconsistent on long contexts. When it grades its own answers (the default), it tends to be generous. Before trusting a gap between two methods, read a few of the questions where they differ with `show_question`, and if it matters, re-run with a stronger judge. The checkpoint keeps each judge's scores separately.
- **~27 answerable questions is a small sample.** Look at the ± table. Per-type tables have 3–8 questions per cell, so read them as hints.
- **The questions were written from random sections, not from the OKF topic pages**, so the OKF methods don't get questions shaped like their own summaries. The reference answers were checked against the chunk text and the bundle, but still read them: if you disagree with one, fix it in `eval_questions.jsonl` and re-run (only the changed question gets re-graded).
- **Block size differs between methods** (section vs chunk), which affects contextual precision more than the other three.
- **Missing: answer correctness.** None of the four compares the answer with the reference. If you want that, DeepEval's `GEval` with a "does the answer contain the reference's key facts?" criterion adds it as a fifth metric.